# 测试用代码

## 片段1：基本信息获取

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import time
import pandas as pd

# 设置 Selenium 选项
options = Options()
options.add_argument("--headless")  # 无头模式
driver = webdriver.Chrome(options=options)

# 打开页面
url = "https://rollcall.com/factbase/"
driver.get(url)

# 模拟滚动页面加载全部内容（可根据实际滚动次数调整）
scroll_pause = 2
last_height = driver.execute_script("return document.body.scrollHeight")
for _ in range(10):  # 控制滚动次数，防止无限滚动
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(scroll_pause)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# 解析 HTML
soup = BeautifulSoup(driver.page_source, "html.parser")
cards = soup.find_all("div", class_="mb-4 border flex gap-4 sm:gap-4")

# 提取信息
data = []
for card in cards:
    try:
        address = card.find("div", class_="hidden md:block font-graphik text-sm sm:text-md text-[#101010] border-r pr-2 border-gray-[#D4D4D4]").text.strip()
    except:
        address = ""

    try:
        date = card.find("div", class_="font-graphik text-sm md:text-md text-[#101010]").text.strip()
    except:
        date = ""

    try:
        title = card.find("div", class_="text-sm font-graphik text-[#2F3C4B] font-semibold mb-2 leading-normal pr-4").text.strip()
    except:
        title = ""

    try:
        link = card.find("a", class_="text-left inline-block p-2 bg-[#015582] text-white text-sm hover:bg-[#2F3C4B] rounded-md font-roboto mt-2 lg:mt-0 relative bottom-2")["href"]
    except:
        link = ""

    data.append([address, date, title, link])

driver.quit()

# 保存或打印结果
df = pd.DataFrame(data, columns=["ADDRESS", "DATE", "TITLE", "LINK"])
print(df.head())  # 打印前几行


## 片段2：详细爬取

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# 假设 df 是之前爬到的 DataFrame，这里取前5个非空 LINK
valid_df = df[df["LINK"] != ""].head(5)

# 如果链接是相对路径，加上前缀
base_url = "https://rollcall.com"
def full_url(link):
    return link if link.startswith("http") else base_url + link

results = []

for _, row in valid_df.iterrows():
    link = full_url(row["LINK"])
    try:
        res = requests.get(link, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        # 找所有 id 以 "dt" 开头的 div（不硬编码 A-B-C-X）
        divs = soup.find_all("div", id=re.compile(r"^dt\d{2}-\d{2}-[a-z0-9]{8}-\d+$"))
        
        for div in divs:
            try:
                speaker = div.find("h2", class_="text-md inline").text.strip()
            except:
                speaker = ""
            try:
                duration = div.find("span", class_="text-xs text-gray-600 inline ml-2").text.strip()
            except:
                duration = ""
            try:
                # 更稳健的方式提取 TEXT
                text_div = div.find("div", class_=lambda c: c and "leading-loose" in c and "text-gray-600" in c)
                text = text_div.text.strip() if text_div else ""
            except:
                text = ""
            
            results.append([
                row["ADDRESS"],
                row["DATE"],
                row["TITLE"],
                link,
                speaker,
                duration,
                text
            ])
    except Exception as e:
        print(f"访问失败：{link}，错误：{e}")

# 构建完整结果 DataFrame
detailed_df = pd.DataFrame(results, columns=[
    "ADDRESS", "DATE", "TITLE", "LINK", "SPEAKER", "DURATION", "TEXT"
])

# 展示前几行确认
print(detailed_df.head())



In [ ]:
# 保存为 CSV 文件
detailed_df.to_csv("factbase_transcripts.csv", index=False, encoding="utf-8-sig")
print("数据已保存为 factbase_transcripts.csv")


## 完整代码（测试）

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time
import re
import os
import string
from tqdm import tqdm

# ---------- 设置 Selenium ----------
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

# ---------- 打开主页面并滚动 ----------
url = "https://rollcall.com/factbase/"
driver.get(url)

scroll_pause = 2
last_height = driver.execute_script("return document.body.scrollHeight")
for _ in range(10):  # 滚动10次
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(scroll_pause)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# ---------- 解析主页面 ----------
soup = BeautifulSoup(driver.page_source, "html.parser")
cards = soup.find_all("div", class_="mb-4 border flex gap-4 sm:gap-4")
driver.quit()

# ---------- 提取前5个含 LINK 的卡片 ----------
data = []
for card in cards:
    try:
        address = card.find("div", class_="hidden md:block font-graphik text-sm sm:text-md text-[#101010] border-r pr-2 border-gray-[#D4D4D4]").text.strip()
    except:
        address = ""
    try:
        date = card.find("div", class_="font-graphik text-sm md:text-md text-[#101010]").text.strip()
    except:
        date = ""
    try:
        title = card.find("div", class_="text-sm font-graphik text-[#2F3C4B] font-semibold mb-2 leading-normal pr-4").text.strip()
    except:
        title = ""
    try:
        link = card.find("a", class_="text-left inline-block p-2 bg-[#015582] text-white text-sm hover:bg-[#2F3C4B] rounded-md font-roboto mt-2 lg:mt-0 relative bottom-2")["href"]
    except:
        link = ""

    if link:
        data.append([address, date, title, link])
    if len(data) >= 5:
        break  # 只取前5个有 LINK 的

# ---------- 函数：清理标题为文件名 ----------
def clean_filename(name):
    valid_chars = "-_.() %s%s" % (string.ascii_letters, string.digits)
    return "".join(c for c in name if c in valid_chars).replace(" ", "_")[:100]  # 截断过长标题

# ---------- 访问每个 LINK 页面 ----------
base_url = "https://rollcall.com"
for entry in tqdm(data, desc="爬取进度", unit="页"):
    address, date, title, link = entry
    full_link = link if link.startswith("http") else base_url + link
    results = []

    try:
        res = requests.get(full_link, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        divs = soup.find_all("div", id=re.compile(r"^dt\d{2}-\d{2}-[a-z0-9]{8}-\d+$"))

        for div in divs:
            try:
                speaker = div.find("h2", class_="text-md inline").text.strip()
            except:
                speaker = ""
            try:
                duration = div.find("span", class_="text-xs text-gray-600 inline ml-2").text.strip()
            except:
                duration = ""
            try:
                text_div = div.find("div", class_=lambda c: c and "leading-loose" in c and "text-gray-600" in c)
                text = text_div.text.strip() if text_div else ""
            except:
                text = ""

            results.append([address, date, title, full_link, speaker, duration, text])

        # 保存为 CSV
        df = pd.DataFrame(results, columns=["ADDRESS", "DATE", "TITLE", "LINK", "SPEAKER", "DURATION", "TEXT"])
        filename = clean_filename(title) + ".csv"
        df.to_csv(filename, index=False, encoding="utf-8-sig")
        
    except Exception as e:
        print(f"\n访问失败：{full_link}，错误：{e}")


# 最终版本

In [ ]:
import os
import re
import string
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------- 配置 ----------
output_dir = "Trump"
log_path = os.path.join(output_dir, "error_log.txt")
index_csv = os.path.join(output_dir, "index.csv")
os.makedirs(output_dir, exist_ok=True)

# ---------- 工具函数 ----------
def clean_filename(name):
    valid_chars = "-_.() %s%s" % (string.ascii_letters, string.digits)
    cleaned = "".join(c for c in name if c in valid_chars).replace(" ", "_")
    return cleaned[:100]

def log_error(message):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(message + "\n")

# ---------- User-Agent 轮换 ----------
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Firefox/113.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 16_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile Safari/605.1.15"
]

# ---------- 使用 Selenium 滚动至所有内容加载完成 ----------
def load_main_page():
    options = Options()
    options.add_argument("--headless")
    driver = webdriver.Chrome(options=options)

    url = "https://rollcall.com/factbase/"
    driver.get(url)

    scroll_pause = 2
    no_change_count = 0
    last_height = driver.execute_script("return document.body.scrollHeight")

    while no_change_count < 2:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            no_change_count += 1
        else:
            no_change_count = 0
            last_height = new_height

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    return soup

# ---------- 提取所有有效条目并保存为 index.csv ----------
def extract_index(soup):
    cards = soup.find_all("div", class_="mb-4 border flex gap-4 sm:gap-4")
    base_url = "https://rollcall.com"
    records = []

    for card in cards:
        try:
            address = card.find("div", class_="hidden md:block font-graphik text-sm sm:text-md text-[#101010] border-r pr-2 border-gray-[#D4D4D4]").text.strip()
            date = card.find("div", class_="font-graphik text-sm md:text-md text-[#101010]").text.strip()
            title = card.find("div", class_="text-sm font-graphik text-[#2F3C4B] font-semibold mb-2 leading-normal pr-4").text.strip()
            link_tag = card.find("a", class_="text-left inline-block p-2 bg-[#015582] text-white text-sm hover:bg-[#2F3C4B] rounded-md font-roboto mt-2 lg:mt-0 relative bottom-2")
            link = link_tag["href"] if link_tag and link_tag.get("href") else ""
            if not link:
                log_error(f"[跳过] 缺少链接: {title}")
                continue
            full_link = link if link.startswith("http") else base_url + link
            filename = clean_filename(title)
            records.append([address, date, title, full_link, filename])
        except Exception as e:
            log_error(f"[解析错误] 主列表某项失败: {e}")

    df = pd.DataFrame(records, columns=["ADDRESS", "DATE", "TITLE", "LINK", "FILENAME"])
    df.drop_duplicates(subset="FILENAME", inplace=True)
    df.to_csv(index_csv, index=False, encoding="utf-8-sig")
    return df

# ---------- 爬取详情页 ----------
def crawl_detail_page(row, index, total):
    address = row["ADDRESS"]
    date = row["DATE"]
    title = row["TITLE"]
    link = row["LINK"]
    filename = row["FILENAME"]
    filepath = os.path.join(output_dir, filename + ".csv")

    if os.path.exists(filepath):
        log_error(f"[跳过] 已存在文件: {title}")
        return

    print(f"[{index}/{total}] 正在爬取：{title}")
    results = []

    time.sleep(random.uniform(0.5, 1.5))
    headers = {"User-Agent": random.choice(USER_AGENTS)}

    try:
        res = requests.get(link, headers=headers, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        divs = soup.find_all("div", id=re.compile(r"^dt\d{2}-\d{2}-[a-z0-9]{8}-\d+$"))

        if not divs:
            log_error(f"[空页面] 无有效内容: {title}")
            return

        for div in divs:
            try:
                speaker = div.find("h2", class_="text-md inline").text.strip()
            except:
                speaker = ""
            try:
                duration = div.find("span", class_="text-xs text-gray-600 inline ml-2").text.strip()
            except:
                duration = ""
            try:
                text_div = div.find("div", class_=lambda c: c and "leading-loose" in c and "text-gray-600" in c)
                text = text_div.text.strip() if text_div else ""
            except:
                text = ""

            results.append([address, date, title, link, speaker, duration, text])

        df = pd.DataFrame(results, columns=["ADDRESS", "DATE", "TITLE", "LINK", "SPEAKER", "DURATION", "TEXT"])
        df.to_csv(filepath, index=False, encoding="utf-8-sig")

    except Exception as e:
        log_error(f"[失败] 页面访问或解析错误: {title} | 错误: {str(e)}")

# ---------- 主流程 ----------
if not os.path.exists(index_csv):
    print("🔄 正在加载一级目录页面...")
    soup = load_main_page()
    print("✅ 页面加载完成，提取文章目录...")
    df_index = extract_index(soup)
    print(f"✅ 共发现 {len(df_index)} 篇文章，目录已保存为 index.csv")
else:
    df_index = pd.read_csv(index_csv, encoding="utf-8-sig")
    print(f"📂 已加载 index.csv，共 {len(df_index)} 篇文章待检查")

# ---------- 并发爬取未完成条目 ----------
rows_to_crawl = [
    (row, idx + 1, len(df_index))
    for idx, row in df_index.iterrows()
    if not os.path.exists(os.path.join(output_dir, row["FILENAME"] + ".csv"))
]

print(f"🚀 开始爬取 {len(rows_to_crawl)} 篇未完成文章")

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [
        executor.submit(crawl_detail_page, row, i, total)
        for row, i, total in rows_to_crawl
    ]
    for _ in as_completed(futures):
        pass


# 后处理，对于爬取失败的文章，重来一遍

In [ ]:
import os
import pandas as pd
import re
import random
import time
import requests
from bs4 import BeautifulSoup

# ---------- 配置 ----------
output_dir = "Trump\index"
log_path = os.path.join(output_dir, "error_log.txt")
index_path = os.path.join(output_dir, "index.csv")

# ---------- User-Agent 列表 ----------
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Firefox/113.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 16_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile Safari/605.1.15"
]

# ---------- 提取失败标题 ----------
with open(log_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

failed_titles = []
for line in lines:
    match = re.search(r"\[失败\] 页面访问或解析错误: (.+?) \| 错误:", line)
    if match:
        failed_titles.append(match.group(1).strip())

print(f"🎯 需要重新爬取 {len(failed_titles)} 篇文章")

# ---------- 加载 index.csv ----------
df_index = pd.read_csv(index_path, encoding="utf-8-sig")

# ---------- 定义重新爬虫函数 ----------
def crawl_detail_page(row):
    address = row["ADDRESS"]
    date = row["DATE"]
    title = row["TITLE"]
    link = row["LINK"]
    filename = row["FILENAME"]
    filepath = os.path.join(output_dir, filename + ".csv")

    print(f"🔁 重新爬取：{title}")
    results = []

    time.sleep(random.uniform(0.5, 1.5))
    headers = {"User-Agent": random.choice(USER_AGENTS)}

    try:
        res = requests.get(link, headers=headers, timeout=15)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        divs = soup.find_all("div", id=re.compile(r"^dt\d{2}-\d{2}-[a-z0-9]{8}-\d+$"))

        if not divs:
            print(f"⚠️  空页面：{title}")
            return

        for div in divs:
            try:
                speaker = div.find("h2", class_="text-md inline").text.strip()
            except:
                speaker = ""
            try:
                duration = div.find("span", class_="text-xs text-gray-600 inline ml-2").text.strip()
            except:
                duration = ""
            try:
                text_div = div.find("div", class_=lambda c: c and "leading-loose" in c and "text-gray-600" in c)
                text = text_div.text.strip() if text_div else ""
            except:
                text = ""

            results.append([address, date, title, link, speaker, duration, text])

        df = pd.DataFrame(results, columns=["ADDRESS", "DATE", "TITLE", "LINK", "SPEAKER", "DURATION", "TEXT"])
        df.to_csv(filepath, index=False, encoding="utf-8-sig")
        print(f"✅ 成功保存：{filename}.csv")

    except Exception as e:
        print(f"❌ 重试失败：{title} | 错误: {e}")

# ---------- 对每个失败标题进行重爬 ----------
for title in failed_titles:
    row_match = df_index[df_index["TITLE"].str.strip() == title]
    if row_match.empty:
        print(f"⚠️  无法在 index.csv 中匹配到标题：{title}")
        continue
    crawl_detail_page(row_match.iloc[0])


In [ ]:
import os
import shutil
import pandas as pd

# ---------- 配置 ----------
source_dir = "Trump"
year_target = "sort_by_year"
category_target = "sort_by_category"

os.makedirs(year_target, exist_ok=True)
os.makedirs(category_target, exist_ok=True)

csv_files = [
    f for f in os.listdir(source_dir)
    if f.endswith(".csv") and f != "index.csv"
]

def extract_year(date_str):
    for token in str(date_str).split():
        if token.isdigit() and len(token) == 4:
            return token
    return None

def extract_category(title_str):
    parts = str(title_str).split(":")
    if len(parts) > 1:
        return parts[0].strip()
    return "Uncategorized"

# ---------- 遍历并复制 ----------
for filename in csv_files:
    filepath = os.path.join(source_dir, filename)
    try:
        df = pd.read_csv(filepath, encoding="utf-8-sig")
        if df.empty or "DATE" not in df.columns or "TITLE" not in df.columns:
            print(f"⚠️ 跳过无效文件: {filename}")
            continue

        # 提取信息
        date = df.iloc[0]["DATE"]
        title = df.iloc[0]["TITLE"]

        year = extract_year(date)
        category = extract_category(title)

        # ---------- 复制到 sort_by_year ----------
        if year:
            year_folder = os.path.join(year_target, year)
            os.makedirs(year_folder, exist_ok=True)
            shutil.copy2(filepath, os.path.join(year_folder, filename))
            print(f"📁 按年份复制到: {year}/{filename}")
        else:
            print(f"⚠️ 无法解析年份: {filename}")

        # ---------- 复制到 sort_by_category ----------
        category_folder = os.path.join(category_target, category)
        os.makedirs(category_folder, exist_ok=True)
        shutil.copy2(filepath, os.path.join(category_folder, filename))
        print(f"📂 按类别复制到: {category}/{filename}")

    except Exception as e:
        print(f"❌ 处理文件失败: {filename} | 错误: {e}")


# 补充爬虫：再次爬取

In [ ]:
import os
import re
import string
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------- 配置 ----------
output_dir = r"C:\Users\numinose\Desktop\Trump in factbase\Complement"
log_path = os.path.join(output_dir, "error_log.txt")
index_csv = os.path.join(output_dir, "index.csv")
os.makedirs(output_dir, exist_ok=True)

def clean_filename(name):
    valid_chars = "-_.() %s%s" % (string.ascii_letters, string.digits)
    cleaned = "".join(c for c in name if c in valid_chars).replace(" ", "_")
    return cleaned[:100]

def log_error(message):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(message + "\n")

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Firefox/113.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 16_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile Safari/605.1.15"
]

# ---------- 优化后的滚动逻辑 ----------
def load_main_page():
    options = Options()
    options.add_argument("--headless")
    driver = webdriver.Chrome(options=options)
    url = "https://rollcall.com/factbase/"
    driver.get(url)

    seen = set()
    no_new_rounds = 0
    repetition_limit = 10
    scroll_pause = 2

    while True:
        time.sleep(scroll_pause)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        cards = soup.find_all("div", class_="mb-4 border flex gap-4 sm:gap-4")

        new_cards = 0
        for card in cards:
            key = str(card)
            if key not in seen:
                seen.add(key)
                new_cards += 1

        print(f"🔄 页面滚动：新增 {new_cards} 个卡片，总计 {len(seen)}")

        if new_cards == 0:
            no_new_rounds += 1
        else:
            no_new_rounds = 0

        if no_new_rounds >= repetition_limit:
            print(f"🛑 连续 {repetition_limit} 次无新增卡片，停止滚动")
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    final_soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    return final_soup

# ---------- 提取一级索引 ----------
def extract_index(soup):
    base_url = "https://rollcall.com"
    cards = soup.find_all("div", class_="mb-4 border flex gap-4 sm:gap-4")
    records = []

    for card in cards:
        try:
            type_div = card.find("div", class_="block font-graphik text-sm sm:text-md text-[#101010] border-r pr-2 border-gray-[#D4D4D4]")
            if not type_div or "Document" not in type_div.text.strip():
                continue

            address = card.find("div", class_="hidden md:block font-graphik text-sm sm:text-md text-[#101010] border-r pr-2 border-gray-[#D4D4D4]").text.strip()
            date = card.find("div", class_="font-graphik text-sm md:text-md text-[#101010]").text.strip()
            title = card.find("div", class_="text-sm font-graphik text-[#2F3C4B] font-semibold mb-2 leading-normal pr-4").text.strip()

            link_tag = card.find("a", href=re.compile(r"^https://rollcall\.com/factbase/"))
            link = link_tag["href"] if link_tag else ""

            if not link:
                log_error(f"[跳过] 无法提取链接: {title}")
                continue

            filename = clean_filename(title)
            records.append([address, date, title, link, filename])

        except Exception as e:
            log_error(f"[解析错误] 主列表某项失败: {e}")

    df = pd.DataFrame(records, columns=["ADDRESS", "DATE", "TITLE", "LINK", "FILENAME"])
    df.drop_duplicates(subset="FILENAME", inplace=True)
    df.to_csv(index_csv, index=False, encoding="utf-8-sig")
    return df

# ---------- 二级页面爬虫 ----------
def crawl_detail_page(row, index, total):
    address = row["ADDRESS"]
    date = row["DATE"]
    title = row["TITLE"]
    link = row["LINK"]
    filename = row["FILENAME"]
    filepath = os.path.join(output_dir, filename + ".csv")

    if os.path.exists(filepath):
        log_error(f"[跳过] 已存在文件: {title}")
        return

    print(f"[{index}/{total}] 正在爬取：{title}")
    time.sleep(random.uniform(0.5, 1.5))
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    results = []

    try:
        res = requests.get(link, headers=headers, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        divs = soup.find_all("div", id=re.compile(r"^dt\d{2}-\d{2}-[a-z0-9]{8}-\d+$"))

        if not divs:
            log_error(f"[结构异常] 无内容: {title}")
            pd.DataFrame([{
                "ADDRESS": address,
                "DATE": date,
                "TITLE": title,
                "LINK": link,
                "SPEAKER": "",
                "TEXT": ""
            }]).to_csv(filepath, index=False, encoding="utf-8-sig")
            return

        for div in divs:
            speaker_tag = div.select_one("h2.text-md.inline")
            text_tag = div.select_one("div.text-gray-600.leading-loose")

            speaker = speaker_tag.text.strip() if speaker_tag else ""
            text = text_tag.text.strip() if text_tag else ""

            results.append({
                "ADDRESS": address,
                "DATE": date,
                "TITLE": title,
                "LINK": link,
                "SPEAKER": speaker,
                "TEXT": text
            })

        pd.DataFrame(results).to_csv(filepath, index=False, encoding="utf-8-sig")

    except Exception as e:
        log_error(f"[失败] 页面访问错误: {title} | 错误: {e}")

# ---------- 主逻辑 ----------
if not os.path.exists(index_csv):
    print("🔄 正在加载一级页面...")
    soup = load_main_page()
    print("✅ 页面加载完成，提取 Document 条目...")
    df_index = extract_index(soup)
    print(f"✅ 共提取 Document 类型 {len(df_index)} 条")
else:
    df_index = pd.read_csv(index_csv, encoding="utf-8-sig")
    print(f"📂 已加载 index.csv，共 {len(df_index)} 条")

rows_to_crawl = [
    (row, idx + 1, len(df_index))
    for idx, row in df_index.iterrows()
    if not os.path.exists(os.path.join(output_dir, row["FILENAME"] + ".csv"))
]

print(f"🚀 开始爬取 {len(rows_to_crawl)} 篇未完成文章...")

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(crawl_detail_page, row, i, total) for row, i, total in rows_to_crawl]
    for _ in as_completed(futures):
        pass

print("✅ 所有任务完成。")
